In [1]:
# ── Cell 1: Diagnose the two failure sources in the test sentence ─────────
import sys, json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
sys.path.insert(0, str(PROJECT_ROOT / "app"))

from corrector import SpellCorrector
from edit_distance import damerau_levenshtein

sc = SpellCorrector(DATA_DIR)

sentence = "I hv a frind sh ws in tje univrsty lts yer"
intended = ["i", "have", "a", "friend", "she", "was", "in", "the",
            "university", "last", "year"]

print("── Token-by-token: why did each word pass or fail? ──")
for typed, truth in zip(sentence.lower().split(), intended):
    in_dict = typed in sc.dictionary
    in_corpus = typed in sc.word_freq
    d = damerau_levenshtein(typed, truth)
    status = "ACCEPTED (invisible!)" if in_dict and typed != truth else \
             "flagged" if not in_dict else "correct"
    print(f"  {typed:<10} → {truth:<11} dist={d}  "
          f"dict={'Y' if in_dict else 'n'}  corpus={'Y' if in_corpus else 'n'}  {status}")

print("\n── Where does the truth rank for the flagged ones? ──")
for typed, truth in [("frind","friend"), ("tje","the"), ("univrsty","university"), ("lts","last")]:
    ranked, tier = sc.suggest(typed, top=10)
    words = [w for w, _ in ranked]
    pos = words.index(truth) + 1 if truth in words else "not in top-10"
    print(f"  {typed:<10} truth '{truth}' at rank: {pos}   top-3: {words[:3]}")

print("\n── How much junk does the coverage layer contain? ──")
short = [w for w in sc.dictionary if len(w) <= 3]
print(f"  Words of length ≤3 in dictionary: {len(short):,}")
print(f"  Sample: {sorted(short)[200:230]}")
suspicious = [w for w in ["hv","sh","ws","yer","tje","frind","teh","lts",
                          "wich","jat","soo","abt","pls","thx"] if w in sc.dictionary]
print(f"  Known-junk probes present: {suspicious}")

── Token-by-token: why did each word pass or fail? ──
  i          → i           dist=0  dict=Y  corpus=Y  correct
  hv         → have        dist=2  dict=Y  corpus=n  ACCEPTED (invisible!)
  a          → a           dist=0  dict=Y  corpus=Y  correct
  frind      → friend      dist=1  dict=n  corpus=n  flagged
  sh         → she         dist=1  dict=Y  corpus=n  ACCEPTED (invisible!)
  ws         → was         dist=1  dict=Y  corpus=n  ACCEPTED (invisible!)
  in         → in          dist=0  dict=Y  corpus=Y  correct
  tje        → the         dist=1  dict=n  corpus=n  flagged
  univrsty   → university  dist=2  dict=n  corpus=n  flagged
  lts        → last        dist=2  dict=n  corpus=n  flagged
  yer        → year        dist=1  dict=Y  corpus=n  ACCEPTED (invisible!)

── Where does the truth rank for the flagged ones? ──
  frind      truth 'friend' at rank: 2   top-3: ['find', 'friend', 'frond']
  tje        truth 'the' at rank: 1   top-3: ['the', 'tie', 'tue']
  univrsty   truth 'u

In [2]:
# ── Cell 2: Measure real-world frequency of our dictionary's contents ─────
from wordfreq import zipf_frequency
import random

random.seed(42)

# 1. The probes: our invisible errors, known junk, real-rare, real-common
probes = {
    "invisible errors": ["hv", "sh", "ws", "yer"],
    "other junk found": ["jat", "abt", "bcd", "bdl", "bbl", "tje"],
    "misspellings (must stay OUT)": ["teh", "wich", "frind", "recieve", "lts"],
    "common real words": ["the", "have", "she", "was", "year", "last", "university"],
    "rare real words (science)": ["aberrant", "inflorescence", "sepal",
                                   "rudimentary", "gneiss", "eucrasy"],
}
print("── Zipf scores (0 = never seen in real text; ~7.7 = 'the') ──")
for group, words in probes.items():
    print(f"\n  {group}:")
    for w in words:
        z = zipf_frequency(w, "en")
        in_dict = "in-dict" if w in sc.dictionary else "not-in-dict"
        print(f"    {w:<14} zipf={z:.2f}   {in_dict}")

# 2. Distribution over the whole coverage layer: how much has zero evidence?
sample = random.sample(sorted(sc.dictionary), 30000)
buckets = {"zipf = 0 (no real usage)": 0, "0 < zipf < 1.5": 0,
           "1.5 – 3 (rare)": 0, "3 – 5 (normal)": 0, "5+ (common)": 0}
for w in sample:
    z = zipf_frequency(w, "en")
    if z == 0: buckets["zipf = 0 (no real usage)"] += 1
    elif z < 1.5: buckets["0 < zipf < 1.5"] += 1
    elif z < 3: buckets["1.5 – 3 (rare)"] += 1
    elif z < 5: buckets["3 – 5 (normal)"] += 1
    else: buckets["5+ (common)"] += 1
print("\n── 30,000-word sample of our dictionary, by real-world usage ──")
for k, v in buckets.items():
    print(f"  {k:<28} {v:>6,}  ({v/len(sample):.1%})")

# 3. The safety check: would a frequency threshold hurt our benchmark?
#    (pairs file is a dict: misspelling -> correction OR list of corrections)
pairs = json.loads((DATA_DIR / "misspelling_pairs.json").read_text())
wrongs = sorted(pairs.keys())
truths = sorted({t for corr in pairs.values()
                 for t in (corr if isinstance(corr, list) else [corr])})
for T in [1.0, 1.5, 2.0, 2.5]:
    truth_lost = sum(zipf_frequency(t, "en") < T for t in truths)
    wrong_alive = sum(zipf_frequency(m, "en") >= T for m in wrongs)
    print(f"\n  threshold T={T}: {truth_lost}/{len(truths)} truth words would be dropped"
          f"  ·  {wrong_alive}/{len(wrongs)} misspellings score ABOVE it")

── Zipf scores (0 = never seen in real text; ~7.7 = 'the') ──

  invisible errors:
    hv             zipf=2.83   in-dict
    sh             zipf=3.86   in-dict
    ws             zipf=3.31   in-dict
    yer            zipf=3.36   in-dict

  other junk found:
    jat            zipf=2.37   in-dict
    abt            zipf=3.24   in-dict
    bcd            zipf=2.34   in-dict
    bdl            zipf=1.88   in-dict
    bbl            zipf=3.05   in-dict
    tje            zipf=1.84   not-in-dict

  misspellings (must stay OUT):
    teh            zipf=3.04   not-in-dict
    wich           zipf=2.77   not-in-dict
    frind          zipf=1.40   not-in-dict
    recieve        zipf=2.75   not-in-dict
    lts            zipf=2.76   not-in-dict

  common real words:
    the            zipf=7.73   in-dict
    have           zipf=6.71   in-dict
    she            zipf=6.26   in-dict
    was            zipf=6.82   in-dict
    year           zipf=5.96   in-dict
    last           zipf=5.86   in-dic

In [3]:
# ── Cell 3: Derive per-word trust from frequency + neighbour shadowing ────
from wordfreq import zipf_frequency

T_TRUST = 4.0   # zipf at/above this: trusted outright   (calibrated next cell)
T_GAP   = 2.0   # neighbour must beat the word by this much to cast suspicion

def strongest_neighbour(w):
    """Most frequent dictionary word within one edit (excluding w itself)."""
    best, best_z = None, -1.0
    for n in sc._known(sc._edits1(w)) - {w}:
        z = zipf_frequency(n, "en")
        if z > best_z:
            best, best_z = n, z
    return best, best_z

def classify(w):
    if w in sc.word_freq:                      # domain corpus veto
        return "trusted (corpus)", None, None
    z = zipf_frequency(w, "en")
    if z >= T_TRUST:
        return "trusted (common)", None, z
    nb, nz = strongest_neighbour(w)
    if nb and nz - z >= T_GAP and nz >= T_TRUST:
        return "SUSPICIOUS", nb, round(nz - z, 2)
    return "quiet-rare", None, z

print("── Probe classifications ──")
for w in ["hv","sh","ws","yer","abt","jat","bcd",          # must be SUSPICIOUS
          "the","have","she","university",                  # must be trusted
          "sepal","gneiss","aberrant","inflorescence",      # corpus veto test
          "eucrasy","bdl"]:                                 # quiet-rare candidates
    tier, nb, info = classify(w)
    extra = f"→ shadowed by '{nb}' (gap {info})" if nb else (f"zipf {info}" if info is not None else "")
    print(f"  {w:<14} {tier:<18} {extra}")

# Proportions on a sample (neighbour search is the slow part — sample, don't sweep)
import random; random.seed(42)
sample = random.sample(sorted(sc.dictionary), 3000)
from collections import Counter
dist = Counter(classify(w)[0] for w in sample)
print("\n── 3,000-word dictionary sample ──")
for k, v in dist.most_common():
    print(f"  {k:<18} {v:>5}  ({v/len(sample):.1%})")

# Safety: benchmark truths must not be suspicious (or engine would fight itself)
pairs = json.loads((DATA_DIR / "misspelling_pairs.json").read_text())
truths = sorted({t for corr in pairs.values()
                 for t in (corr if isinstance(corr, list) else [corr])})
sus_truths = [t for t in truths if t in sc.dictionary and classify(t)[0] == "SUSPICIOUS"]
print(f"\n── Benchmark safety: {len(sus_truths)}/{len(truths)} truth words "
      f"classified SUSPICIOUS ──")
print("  Sample:", sus_truths[:15])

── Probe classifications ──
  hv             SUSPICIOUS         → shadowed by 'he' (gap 3.86)
  sh             SUSPICIOUS         → shadowed by 'so' (gap 2.66)
  ws             SUSPICIOUS         → shadowed by 'is' (gap 3.76)
  yer            SUSPICIOUS         → shadowed by 'her' (gap 2.94)
  abt            SUSPICIOUS         → shadowed by 'at' (gap 3.46)
  jat            SUSPICIOUS         → shadowed by 'at' (gap 4.33)
  bcd            SUSPICIOUS         → shadowed by 'bad' (gap 3.19)
  the            trusted (corpus)   
  have           trusted (corpus)   
  she            trusted (corpus)   
  university     trusted (corpus)   
  sepal          SUSPICIOUS         → shadowed by 'seal' (gap 2.49)
  gneiss         quiet-rare         zipf 2.33
  aberrant       trusted (corpus)   
  inflorescence  quiet-rare         zipf 2.6
  eucrasy        quiet-rare         zipf 0.0
  bdl            quiet-rare         zipf 1.88

── 3,000-word dictionary sample ──
  quiet-rare          2741  (91.4%)
 

In [4]:
# ── Cell 4: Precompute the full trust map and save it as an artefact ──────
import time

t0 = time.time()
suspicious = {}          # word -> top alternatives [(alt, zipf), ...]
counts = {"trusted_corpus": 0, "trusted_common": 0, "quiet_rare": 0, "suspicious": 0}

words = sorted(sc.dictionary)
for i, w in enumerate(words):
    if i % 50000 == 0:
        print(f"  {i:>7,}/{len(words):,}  ({time.time()-t0:>5.0f}s)  "
              f"suspicious so far: {counts['suspicious']:,}")
    if w in sc.word_freq:
        counts["trusted_corpus"] += 1
        continue
    z = zipf_frequency(w, "en")
    if z >= T_TRUST:
        counts["trusted_common"] += 1
        continue
    neigh = [(n, zipf_frequency(n, "en")) for n in sc._known(sc._edits1(w)) - {w}]
    strong = sorted((p for p in neigh if p[1] >= T_TRUST and p[1] - z >= T_GAP),
                    key=lambda p: -p[1])
    if strong:
        counts["suspicious"] += 1
        suspicious[w] = [(n, round(nz, 2)) for n, nz in strong[:5]]
    else:
        counts["quiet_rare"] += 1

print(f"\nDone in {time.time()-t0:.0f}s")
for k, v in counts.items():
    print(f"  {k:<16} {v:>8,}  ({v/len(words):.1%})")

artefact = {
    "thresholds": {"T_TRUST": T_TRUST, "T_GAP": T_GAP},
    "source": "derived from corpus veto + wordfreq zipf + edit-1 shadowing",
    "counts": counts,
    "suspicious": {w: alts for w, alts in suspicious.items()},
}
out = DATA_DIR / "word_trust.json"
out.write_text(json.dumps(artefact))
print(f"\nSaved {out.name}  ({out.stat().st_size/1e6:.1f} MB)")

# Verify the invisible errors are in the artefact, with their alternatives
reloaded = json.loads(out.read_text())
for w in ["hv", "sh", "ws", "yer"]:
    print(f"  {w:<5} → alternatives: {[a for a, _ in reloaded['suspicious'][w]][:3]}")

        0/376,407  (    0s)  suspicious so far: 0
   50,000/376,407  (    4s)  suspicious so far: 2,839
  100,000/376,407  (    8s)  suspicious so far: 5,610
  150,000/376,407  (   12s)  suspicious so far: 8,557
  200,000/376,407  (   15s)  suspicious so far: 11,453
  250,000/376,407  (   20s)  suspicious so far: 13,193
  300,000/376,407  (   23s)  suspicious so far: 16,195
  350,000/376,407  (   28s)  suspicious so far: 18,506

Done in 30s
  trusted_corpus      8,725  (2.3%)
  trusted_common      3,402  (0.9%)
  quiet_rare        344,208  (91.4%)
  suspicious         20,072  (5.3%)

Saved word_trust.json  (0.8 MB)
  hv    → alternatives: ['he', 'tv', 'h']
  sh    → alternatives: ['so', 'she', 's']
  ws    → alternatives: ['is', 'was', 'as']
  yer   → alternatives: ['her', 'year', 'yet']


In [6]:
# ── Cell 5: V2 suspicious-word checking + M_SUS calibration sweep ─────────
import importlib, corrector
importlib.reload(corrector)

class SpellCorrectorV2(corrector.SpellCorrector):
    """Phase 0: adds trust-map checking of 'suspicious' dictionary words."""
    def __init__(self, data_dir, m_sus=4.0, **kw):
        super().__init__(data_dir, **kw)
        trust = json.loads((Path(data_dir) / "word_trust.json").read_text())
        self.suspicious = {w: [a for a, _ in alts]
                           for w, alts in trust["suspicious"].items()}
        self.m_sus = m_sus

    def _check_suspicious(self, word, left, right):
        """Blend-scored hearing: frequency keeps a voice when context is broken."""
        cands = [word] + self.suspicious[word]
        ranking = sorted(((w, self._ctx_logp_blend(w, left, right)) for w in cands),
                         key=lambda x: -x[1])
        s_word = dict(ranking)[word]
        best_alt, s_best = next((w, s) for w, s in ranking if w != word)
        gap = s_best - s_word
        return gap, best_alt, [(w, s) for w, s in ranking if w != word]

    def analyze(self, text, top=5):
        results = super().analyze(text, top)
        for i, r in enumerate(results):
            tok = r["token"]
            if r["status"] == "ok" and tok in self.suspicious:
                left  = results[i-1]["token"] if i > 0 else None
                right = results[i+1]["token"] if i < len(results)-1 else None
                gap, best, ranking = self._check_suspicious(tok, left, right)
                if gap > self.m_sus:
                    r.update(status="suspicious_word", suggestions=ranking[:top],
                             gap=round(gap, 1))
        return results

v2 = SpellCorrectorV2(DATA_DIR)

# ── Probe A: the test sentence — these gaps should be LARGE ──
sent = "I hv a frind sh ws in tje univrsty lts yer"
toks = sent.lower().split()
print("── Probe A: invisible errors (must flag) ──")
gaps_A = {}
for i, t in enumerate(toks):
    if t in v2.suspicious:
        gap, best, _ = v2._check_suspicious(
            t, toks[i-1] if i else None, toks[i+1] if i < len(toks)-1 else None)
        gaps_A[t] = gap
        print(f"  {t:<5} gap={gap:>6.1f}   best alternative: {best}")

# ── Probe B: rare-but-legitimate words in honest contexts (must NOT flag) ──
protect = [("the sepal of the flower", "sepal"),
           ("a positron is emitted", "positron"),
           ("shades of cyan and blue", "cyan"),
           ("to allot the shares fairly", "allot"),
           ("the servile manner of the clerk", "servile")]
print("\n── Probe B: legitimate rare words (must stay quiet) ──")
gaps_B = {}
for sent_b, target in protect:
    tb = sent_b.split()
    i = tb.index(target)
    if target not in v2.suspicious:
        print(f"  {target:<9} not suspicious — no hearing needed"); continue
    gap, best, _ = v2._check_suspicious(
        target, tb[i-1] if i else None, tb[i+1] if i < len(tb)-1 else None)
    gaps_B[target] = gap
    print(f"  {target:<9} gap={gap:>6.1f}   would-be alternative: {best}")

# ── The sweep: pick M_SUS from evidence ──
print("\n── Margin sweep ──")
print(f"  {'M_SUS':<7}{'errors caught (of ' + str(len(gaps_A)) + ')':<24}"
      f"false alarms (of {len(gaps_B)})")
for m in [1, 2, 3, 4, 5, 6, 8, 10]:
    caught = sum(g > m for g in gaps_A.values())
    false  = sum(g > m for g in gaps_B.values())
    print(f"  {m:<7}{caught:<24}{false}")

── Probe A: invisible errors (must flag) ──
  hv    gap=   6.7   best alternative: he
  sh    gap=   5.1   best alternative: so
  ws    gap=   9.3   best alternative: as
  yer   gap=   1.6   best alternative: yet

── Probe B: legitimate rare words (must stay quiet) ──
  sepal     gap=   6.0   would-be alternative: seal
  positron  gap=   7.5   would-be alternative: position
  cyan      gap=  12.1   would-be alternative: can
  allot     gap=  13.4   would-be alternative: allow
  servile   gap=   3.8   would-be alternative: service

── Margin sweep ──
  M_SUS  errors caught (of 4)    false alarms (of 5)
  1      4                       5
  2      3                       5
  3      3                       5
  4      3                       4
  5      3                       4
  6      2                       3
  8      1                       2
  10     0                       2


In [7]:
# ── Cell 6: Corroboration-gated suspicious check + full-sentence verdict ──
class SpellCorrectorV2(corrector.SpellCorrector):
    """Phase 0 final: suspicious words are flagged only when (a) the local
    sentence corroborates (a neighbouring token is itself broken) and
    (b) the blend hearing shows a frequency/context gap."""
    def __init__(self, data_dir, m_sus=1.0, **kw):
        super().__init__(data_dir, **kw)
        trust = json.loads((Path(data_dir) / "word_trust.json").read_text())
        self.suspicious = {w: [a for a, _ in alts]
                           for w, alts in trust["suspicious"].items()}
        self.m_sus = m_sus

    def _check_suspicious(self, word, left, right):
        cands = [word] + self.suspicious[word]
        ranking = sorted(((w, self._ctx_logp_blend(w, left, right)) for w in cands),
                         key=lambda x: -x[1])
        s_word = dict(ranking)[word]
        best_alt, s_best = next((w, s) for w, s in ranking if w != word)
        return s_best - s_word, best_alt, [(w, s) for w, s in ranking if w != word]

    def analyze(self, text, top=5):
        results = super().analyze(text, top)
        broken = [r["status"] == "non_word" or r["token"] in self.suspicious
                  for r in results]
        for i, r in enumerate(results):
            if r["status"] != "ok" or r["token"] not in self.suspicious:
                continue
            corroborated = (i > 0 and broken[i-1]) or \
                           (i < len(results)-1 and broken[i+1])
            if not corroborated:
                continue                      # deferred to the Phase-2 LM hearing
            left  = results[i-1]["token"] if i > 0 else None
            right = results[i+1]["token"] if i < len(results)-1 else None
            gap, best, ranking = self._check_suspicious(r["token"], left, right)
            if gap > self.m_sus:
                r.update(status="suspicious_word", suggestions=ranking[:top],
                         gap=round(gap, 1))
        return results

v2 = SpellCorrectorV2(DATA_DIR)

# ── Probe B first: the innocents must stay silent ──
protect = ["the sepal of the flower", "a positron is emitted",
           "shades of cyan and blue", "to allot the shares fairly",
           "the servile manner of the clerk"]
false_alarms = 0
for s in protect:
    flags = [r["token"] for r in v2.analyze(s) if r["status"] != "ok"]
    false_alarms += len(flags)
    print(f"  {'CLEAN' if not flags else 'FLAGGED!':<9} {s}   {flags}")
print(f"  → false alarms: {false_alarms}")

# ── The test sentence, end to end ──
sent = "I hv a frind sh ws in tje univrsty lts yer"
print(f"\n── {sent!r} ──")
for r in v2.analyze(sent):
    if r["status"] == "ok":
        print(f"  {r['token']:<10} ok")
    else:
        best = r["suggestions"][0][0] if r["suggestions"] else "—"
        gap = f"  gap={r['gap']}" if "gap" in r else ""
        print(f"  {r['token']:<10} {r['status']:<16} → {best}{gap}")

caught = sum(r["status"] != "ok" for r in v2.analyze(sent))
print(f"\n  Errors visible to the engine: {caught}/8 planted "
      f"(v1 saw 4; 'hv' deferred to Phase 2 by design)")

# ── Regression: the old machinery must be untouched ──
old = [r["status"] for r in v2.analyze("we walked acress the field and observed teh birds threw the mist")]
print("\n  Regression (v1 sentence):", old)

  CLEAN     the sepal of the flower   []
  CLEAN     a positron is emitted   []
  CLEAN     shades of cyan and blue   []
  CLEAN     to allot the shares fairly   []
  CLEAN     the servile manner of the clerk   []
  → false alarms: 0

── 'I hv a frind sh ws in tje univrsty lts yer' ──
  i          ok
  hv         ok
  a          ok
  frind      non_word         → find
  sh         suspicious_word  → so  gap=5.1
  ws         suspicious_word  → as  gap=9.3
  in         ok
  tje        non_word         → the
  univrsty   non_word         → university
  lts        non_word         → its
  yer        suspicious_word  → yet  gap=1.6

  Errors visible to the engine: 7/8 planted (v1 saw 4; 'hv' deferred to Phase 2 by design)

  Regression (v1 sentence): ['ok', 'ok', 'non_word', 'ok', 'ok', 'ok', 'ok', 'non_word', 'ok', 'ok', 'ok', 'ok']


In [8]:
# ── Cell 7: Export the V2 corrector and verify end to end ─────────────────
MODULE_SOURCE = '''"""
SpellCorrector — the SciSpell correction engine (V2).
Developed in notebooks/04_Correction_Engine.ipynb and notebooks/08_V2_Dictionary.ipynb.
V2 adds trust-map checking: dictionary words classified suspicious (data-derived,
see word_trust.json) are context-checked when the surrounding sentence corroborates.
"""
import json
import math
from pathlib import Path

from edit_distance import tokenize, weighted_edit_distance
from language_model import BigramLM

ALPHABET = "abcdefghijklmnopqrstuvwxyz'"

class SpellCorrector:
    def __init__(self, data_dir, lam=3.0, m_sus=1.0):
        data_dir = Path(data_dir)
        self.lam = lam
        self.m_sus = m_sus
        self.dictionary = set(
            (data_dir / "dictionary.txt").read_text(encoding="utf-8").split("\\n"))
        self.word_freq = {w: int(c) for w, c in json.loads(
            (data_dir / "word_freq.json").read_text(encoding="utf-8")).items()}
        self.lm = BigramLM(data_dir / "language_model.json")
        conf = json.loads((data_dir / "confusion_sets.json").read_text(encoding="utf-8"))
        self.margin_sym  = conf["margins"]["symmetric"]
        self.margin_asym = conf["margins"]["asymmetric"]
        self.confusable = {w: (set(a), self.margin_sym)
                           for w, a in conf["symmetric"].items()}
        self.confusable.update({w: ({a}, self.margin_asym)
                                for w, a in conf["asymmetric"].items()})
        # V2 trust map — optional artefact; engine runs v1-identically without it
        trust_path = data_dir / "word_trust.json"
        if trust_path.exists():
            trust = json.loads(trust_path.read_text(encoding="utf-8"))
            self.suspicious = {w: [a for a, _ in alts]
                               for w, alts in trust["suspicious"].items()}
        else:
            self.suspicious = {}

    # ── candidate generation ──
    def _edits1(self, word):
        splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]
        return ({L + R[1:] for L, R in splits if R} |
                {L + R[1] + R[0] + R[2:] for L, R in splits if len(R) > 1} |
                {L + c + R[1:] for L, R in splits if R for c in ALPHABET} |
                {L + c + R for L, R in splits for c in ALPHABET})

    def _known(self, strings):
        return {s for s in strings if s in self.dictionary}

    def candidates(self, word):
        if word in self.dictionary:
            return {word}, 0
        c1 = self._known(self._edits1(word))
        if c1:
            return c1, 1
        c2 = self._known({e2 for e1 in self._edits1(word) for e2 in self._edits1(e1)})
        return (c2, 2) if c2 else (set(), -1)

    # ── scoring ──
    def _ctx_logp(self, w, left, right):
        """Pure bigram context — used by the calibrated real-word margin test."""
        s = 0.0
        if left  is not None: s += math.log(self.lm.p_bigram(left, w))
        if right is not None: s += math.log(self.lm.p_bigram(w, right))
        if left is None and right is None: s += math.log(self.lm.p_unigram(w))
        return s

    def _ctx_logp_blend(self, w, left, right):
        """Jelinek-Mercer 0.7 bigram + 0.3 unigram — suggestion ranking and
        suspicious hearings: frequency keeps a voice when context is unknown."""
        s = 0.0
        if left is not None:
            s += math.log(0.7 * self.lm.p_bigram(left, w)
                          + 0.3 * self.lm.p_unigram(w))
        if right is not None:
            s += math.log(0.7 * self.lm.p_bigram(w, right)
                          + 0.3 * self.lm.p_unigram(w))
        if left is None and right is None:
            s += math.log(self.lm.p_unigram(w))
        return s

    def suggest(self, word, left=None, right=None, top=5):
        """Tiered noisy-channel ranking (tier-2 fills spare top-k capacity)."""
        if word in self.dictionary:
            return [(word, 0.0)], 0
        def rank(cands):
            return sorted(((w, -self.lam * weighted_edit_distance(word, w)
                            + self._ctx_logp_blend(w, left, right)) for w in cands),
                          key=lambda x: (-x[1], x[0]))
        c1 = self._known(self._edits1(word))
        ranked = rank(c1)
        tier = 1 if c1 else -1
        if len(ranked) < top:
            c2 = self._known({e2 for e1 in self._edits1(word)
                              for e2 in self._edits1(e1)}) - c1
            ranked += rank(c2)
            if not c1:
                tier = 2 if c2 else -1
        return ranked[:top], tier

    def _check_confusable(self, word, left, right):
        alts, margin = self.confusable[word]
        ranking = sorted(((w, self._ctx_logp(w, left, right))
                          for w in {word} | alts), key=lambda x: -x[1])
        s_word = dict(ranking)[word]
        best_alt, s_best = next((w, s) for w, s in ranking if w != word)
        if s_best - s_word > margin:
            return True, best_alt, ranking
        return False, None, ranking

    def _check_suspicious(self, word, left, right):
        """Blend-scored hearing for trust-demoted words (V2)."""
        cands = [word] + self.suspicious[word]
        ranking = sorted(((w, self._ctx_logp_blend(w, left, right)) for w in cands),
                         key=lambda x: -x[1])
        s_word = dict(ranking)[word]
        best_alt, s_best = next((w, s) for w, s in ranking if w != word)
        return s_best - s_word, best_alt, [(w, s) for w, s in ranking if w != word]

    # ── full pipeline ──
    def analyze(self, text, top=5):
        """One result dict per token: status ok / non_word / real_word_error /
        suspicious_word (V2), with ranked suggestions where flagged."""
        tokens = tokenize(text)
        results = []
        for i, tok in enumerate(tokens):
            left  = tokens[i - 1] if i > 0 else None
            right = tokens[i + 1] if i < len(tokens) - 1 else None
            if tok not in self.dictionary:
                ranked, tier = self.suggest(tok, left, right, top)
                results.append({"token": tok, "status": "non_word",
                                "suggestions": ranked, "tier": tier})
            elif tok in self.confusable:
                flagged, best, ranking = self._check_confusable(tok, left, right)
                results.append({"token": tok,
                                "status": "real_word_error" if flagged else "ok",
                                "suggestions": [(w, s) for w, s in ranking
                                                 if w != tok] if flagged else [],
                                "tier": 0})
            else:
                results.append({"token": tok, "status": "ok",
                                "suggestions": [], "tier": 0})
        # V2 pass: corroboration-gated suspicious hearings
        if self.suspicious:
            broken = [r["status"] == "non_word" or r["token"] in self.suspicious
                      for r in results]
            for i, r in enumerate(results):
                if r["status"] != "ok" or r["token"] not in self.suspicious:
                    continue
                corroborated = (i > 0 and broken[i-1]) or \
                               (i < len(results)-1 and broken[i+1])
                if not corroborated:
                    continue          # deferred to the modern-LM hearing (Phase 2)
                left  = results[i-1]["token"] if i > 0 else None
                right = results[i+1]["token"] if i < len(results)-1 else None
                gap, best, ranking = self._check_suspicious(r["token"], left, right)
                if gap > self.m_sus:
                    r.update(status="suspicious_word", suggestions=ranking[:top],
                             gap=round(gap, 1))
        return results
'''
CORRECTOR_PATH = PROJECT_ROOT / "app" / "corrector.py"
CORRECTOR_PATH.write_text(MODULE_SOURCE, encoding="utf-8")

importlib.reload(corrector)
sc2 = corrector.SpellCorrector(DATA_DIR)

sent = "I hv a frind sh ws in tje univrsty lts yer"
protect = ["the sepal of the flower", "a positron is emitted",
           "shades of cyan and blue", "to allot the shares fairly",
           "the servile manner of the clerk"]
nb_statuses  = [r["status"] for r in v2.analyze(sent)]
mod_statuses = [r["status"] for r in sc2.analyze(sent)]

checks = [
    ("Trust map loaded", len(sc2.suspicious) == 20072, f"{len(sc2.suspicious):,} words"),
    ("Module matches notebook on test sentence", mod_statuses == nb_statuses,
     f"{sum(s != 'ok' for s in mod_statuses)}/8 flagged"),
    ("Protected sentences stay clean",
     all(all(r["status"] == "ok" for r in sc2.analyze(s)) for s in protect),
     f"{len(protect)} sentences"),
    ("v1 regression sentence unchanged",
     [r["status"] for r in sc2.analyze("we walked acress the field and observed teh birds threw the mist")]
     == [r["status"] for r in v2.analyze("we walked acress the field and observed teh birds threw the mist")],
     "statuses equal"),
    ("soo \u2192 so regression holds",
     sc2.suggest("soo", "hem", "hey")[0][0][0] == "so", "named check"),
]
width = max(len(c[0]) for c in checks)
print("VERIFICATION\n" + "\u2500" * (width + 30))
for label, ok, detail in checks:
    print(f"  {'PASS' if ok else 'FAIL'}  {label:<{width}}  {detail}")
print("\u2500" * (width + 30))
print(f"  {sum(ok for _, ok, _ in checks)}/{len(checks)} checks passed")
assert all(ok for _, ok, _ in checks)

VERIFICATION
──────────────────────────────────────────────────────────────────────
  PASS  Trust map loaded                          20,072 words
  PASS  Module matches notebook on test sentence  7/8 flagged
  PASS  Protected sentences stay clean            5 sentences
  PASS  v1 regression sentence unchanged          statuses equal
  PASS  soo → so regression holds                 named check
──────────────────────────────────────────────────────────────────────
  5/5 checks passed
